# 28. 22b 피처로 XGB 다시 튜닝 (H28)

## 배경
- 지금 XGB 설정(depth 3, lr 0.1, 200그루, colsample 0.3)은 **10번에서 예전 피처(any + stats)로 고른 값**
- 22b에서 피처가 바뀜 (any + ratio + 치환 비율 상위 100) → 최적 설정도 달라졌을 수 있음
- 튜닝은 10번에서 LB로 이어진 적 있음 (0.2995 → 0.3206)
- 후보 C(22b XGB 0.7 + LR 0.3)가 LB 0.3650, 시드 5개 평균 0.3660

## 가설
- H28: 22b 피처에서 XGB 설정을 다시 고르면 C(XGB 0.7 + LR 0.3)가 오른다

## 검증 방법
- 기준: 10 설정 (depth 3, colsample 0.3, 200그루), 시드 42
- **한 번에 하나씩** 바꿈: depth 2 / depth 4 / colsample 0.2 / colsample 0.5 / 나무 수 (100~300)
  - 나무 수는 한 번 학습한 모델에서 앞쪽 나무만 써서 예측 (`iteration_range`) → 같은 모델로 여러 나무 수를 비교
- 치환 상위 100개 선택은 22와 같은 `TopNSubs`로 fold 학습 부분에서만
- 비교: XGB 단독 CV와 **섞기 CV (XGB 0.7 + LR 0.3, C와 같은 방식)** 를 fold별로 기준과 비교
- 채택 기준: 섞기가 기준 C보다 5 fold 중 4개 이상 상승
- ⚠️ 여러 설정 중 CV가 가장 좋은 것을 고르면 점수가 조금 낙관적임 → 상승폭이 fold 간 std보다 작으면 채택하지 않음
- test는 예측에만 사용. 제출 파일은 설아님 Windows 데스크탑에서만 생성 (`SAVE`)


In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, run_cv, macro_f1, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
FOLDS = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

def sub_ratio_matrix(df):   # 22와 동일
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):   # 22와 동일: fold 학습 부분에서 상위 n개 치환 열만
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
X22b    = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1)
X22b_te = pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)
print('22b 피처:', X22b.shape)

Linux | CPU x86_64 | xgboost 3.4.1


22b 피처: (6201, 4632)


#### 설정별 학습 (나무 수는 같은 모델에서 단계별로 예측)
- 확률은 `submissions/cache28/`에 저장 (git 제외). 캐시가 없으면 전체 약 50분

In [2]:
CACHE = Path('../submissions/cache28'); CACHE.mkdir(parents=True, exist_ok=True)
BASE = dict(max_depth=3, learning_rate=0.1, colsample_bytree=0.3, tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
CONFIGS = {   # 이름: (기준에서 바꾼 것, 학습할 나무 수, 평가할 나무 수 목록)
    'base (d3, cs0.3)': ({}, 300, [100, 150, 200, 300]),
    'depth 2':          ({'max_depth': 2}, 300, [200, 300]),
    'depth 4':          ({'max_depth': 4}, 200, [100, 150, 200]),
    'colsample 0.2':    ({'colsample_bytree': 0.2}, 300, [200, 300]),
    'colsample 0.5':    ({'colsample_bytree': 0.5}, 200, [150, 200]),
}
K = len(le.classes_)

def run_staged(name, change, n_trees, stages):
    key = name.replace(' ', '_').replace('(', '').replace(')', '').replace(',', '')
    p = CACHE / (key + '.npz')
    if p.exists():
        z = np.load(p); return {s: (z[f'oof_{s}'], z[f'test_{s}']) for s in stages}
    t = time.time()
    oof = {s: np.zeros((len(y), K), np.float32) for s in stages}
    te  = {s: np.zeros((len(test), K), np.float32) for s in stages}
    for k, (tr, va) in enumerate(FOLDS):
        pf = CACHE / f'{key}_fold{k}.npz'          # fold별 중간 저장 (중간에 끊겨도 이어서 실행)
        if pf.exists():
            z = np.load(pf); pv, pt = {s: z[f'va_{s}'] for s in stages}, {s: z[f'te_{s}'] for s in stages}
        else:
            sel = TopNSubs(100).fit(X22b.iloc[tr])
            Xa, Xv, Xt = sel.transform(X22b.iloc[tr]), sel.transform(X22b.iloc[va]), sel.transform(X22b_te)
            m = xgb.XGBClassifier(**{**BASE, **change}, n_estimators=n_trees)
            m.fit(Xa, y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
            pv = {s: m.predict_proba(Xv, iteration_range=(0, s)) for s in stages}
            pt = {s: m.predict_proba(Xt, iteration_range=(0, s)) for s in stages}
            np.savez(pf, **{f'va_{s}': pv[s] for s in stages}, **{f'te_{s}': pt[s] for s in stages})
        for s in stages:
            oof[s][va] = pv[s]; te[s] += pt[s] / len(FOLDS)
    np.savez(p, **{f'oof_{s}': oof[s] for s in stages}, **{f'test_{s}': te[s] for s in stages})
    print(f'  {name}: 학습 {time.time() - t:.0f}s')
    return {s: (oof[s], te[s]) for s in stages}

R = {}
for name, (change, n_trees, stages) in CONFIGS.items():
    for s, v in run_staged(name, change, n_trees, stages).items():
        R[f'{name}, {s}그루'] = v

p = CACHE / 'LR_any.npz'
if p.exists():
    z = np.load(p); LR_oof, LR_te = z['oof'], z['test']
else:
    r = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
               F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=FOLDS, verbose=False)
    LR_oof, LR_te = r['oof_proba'], r['test_proba']; np.savez(p, oof=LR_oof, test=LR_te)
print('설정 수:', len(R))

  base (d3, cs0.3): 학습 0s
  depth 2: 학습 0s
  depth 4: 학습 0s
  colsample 0.2: 학습 0s
  colsample 0.5: 학습 0s


설정 수: 13


#### 비교: XGB 단독과 섞기(XGB 0.7 + LR 0.3), 기준 = 10 설정 200그루

In [3]:
fold_scores = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in FOLDS])
REF = 'base (d3, cs0.3), 200그루'
blend = lambda k, test=False: 0.7 * R[k][1 if test else 0] + 0.3 * (LR_te if test else LR_oof)
ref_x, ref_b = fold_scores(R[REF][0]), fold_scores(blend(REF))
rows = []
for k in R:
    sx, sb = fold_scores(R[k][0]), fold_scores(blend(k))
    d = sb - ref_b
    rows.append({'설정': k, 'XGB 단독 CV': sx.mean(), 'XGB 기준 대비': sx.mean() - ref_x.mean(),
                 '섞기 CV': sb.mean(), '섞기 std': sb.std(), '섞기 기준 대비': d.mean(), '기준보다 높은 fold': f'{(d > 0).sum()}/5',
                 '채택': '✅' if k != REF and (d > 0).sum() >= 4 and d.mean() > ref_b.std() else ''})
res = pd.DataFrame(rows).set_index('설정')
print(f'기준 C (시드 42): 섞기 CV {ref_b.mean():.4f} ± {ref_b.std():.4f} | fold별 {ref_b.round(4)}')
res.round(4)

기준 C (시드 42): 섞기 CV 0.4613 ± 0.0094 | fold별 [0.4567 0.452  0.468  0.4535 0.4765]


,XGB 단독 CV,XGB 기준 대비,섞기 CV,섞기 std,섞기 기준 대비,기준보다 높은 fold,채택
설정,,,,,,,
"base (d3, cs0.3), 100그루",0.4447,-0.0043,0.4568,0.0109,-0.0045,1/5,
"base (d3, cs0.3), 150그루",0.4539,0.0050,0.4616,0.0093,0.0003,2/5,
"base (d3, cs0.3), 200그루",0.4489,0.0000,0.4613,0.0094,0.0000,0/5,
"base (d3, cs0.3), 300그루",0.4509,0.0020,0.4588,0.0099,-0.0026,1/5,
"depth 2, 200그루",0.4487,-0.0003,0.4594,0.0119,-0.0020,2/5,
"depth 2, 300그루",0.4532,0.0043,0.4595,0.0100,-0.0019,2/5,
"depth 4, 100그루",0.4454,-0.0036,0.4562,0.0067,-0.0051,2/5,
"depth 4, 150그루",0.4473,-0.0017,0.4582,0.0060,-0.0032,2/5,
"depth 4, 200그루",0.4493,0.0004,0.4560,0.0085,-0.0053,1/5,


#### 진단 (채택 판단에는 사용하지 않음): 섞기 CV 상위 설정의 test 예측

In [4]:
top = res.drop(index=REF).sort_values('섞기 CV', ascending=False).head(3).index.tolist()
dist = lambda p: pd.Series(le.inverse_transform(p.argmax(1))).value_counts(normalize=True) * 100
c_o, c_t = blend(REF), blend(REF, True)
diag = [{'설정': k, '기준과 다른 환자 OOF %': (blend(k).argmax(1) != c_o.argmax(1)).mean() * 100,
         '기준과 다른 환자 test %': (blend(k, True).argmax(1) != c_t.argmax(1)).mean() * 100} for k in top]
print(pd.DataFrame(diag).set_index('설정').round(1))
pd.DataFrame({'train': train.SUBCLASS.value_counts(normalize=True) * 100, 'C (기준)': dist(c_t),
              **{k: dist(blend(k, True)) for k in top}}).fillna(0).sort_values('C (기준)', ascending=False).round(1).head(6)

                         기준과 다른 환자 OOF %  기준과 다른 환자 test %
설정                                                        
colsample 0.2, 200그루                10.7               6.4
base (d3, cs0.3), 150그루              5.7               4.3
depth 2, 300그루                      11.3               8.9


,train,C (기준),"colsample 0.2, 200그루","base (d3, cs0.3), 150그루","depth 2, 300그루"
STES,6.1,21.4,20.6,20.9,18.7
COAD,3.6,13.8,14.5,14.3,16.3
KIPAN,8.3,5.7,5.5,5.3,5.4
BRCA,12.7,5.1,5.3,5.0,5.0
OV,4.1,4.9,4.8,4.9,4.9
PCPG,2.4,3.6,3.7,3.9,3.7


#### 제출 파일 (채택된 설정만, 설아님 Windows 데스크탑에서 `SAVE = True`)

In [5]:
SAVE = False   # 팀 규칙: 제출 파일은 Windows 데스크탑에서만 생성
adopted = res.index[res['채택'] == '✅'].tolist()
print('채택:', adopted or '없음 (기준보다 4/5 fold 이상 + 상승폭이 std보다 큰 설정 없음)')
if SAVE and adopted:
    best = res.loc[adopted, '섞기 CV'].idxmax()
    tag = best.replace(' ', '').replace('(', '').replace(')', '').replace(',', '_')
    print('저장:', save_submission(submission, blend(best, True), le, name=f'blend_ratio22sub100_retune_{tag}_0.7_LR0.3', cv_score=res.loc[best, '섞기 CV']))

채택: 없음 (기준보다 4/5 fold 이상 + 상승폭이 std보다 큰 설정 없음)


## 결과 (Linux 실행, 시드 42)

기준 C: 10 설정(depth 3, colsample 0.3, 200그루) XGB 0.7 + LR 0.3 = **섞기 CV 0.4613 ± 0.0094**

| 설정 | XGB 단독 CV | 섞기 CV | 섞기 기준 대비 | 기준보다 높은 fold |
| :-- | :-: | :-: | :-: | :-: |
| 기준 (d3, cs 0.3), 200그루 | 0.4489 | 0.4613 | — | — |
| 기준, 100그루 | 0.4447 | 0.4568 | −0.0045 | 1/5 |
| 기준, 150그루 | 0.4539 | 0.4616 | +0.0003 | 2/5 |
| 기준, 300그루 | 0.4509 | 0.4588 | −0.0026 | 1/5 |
| depth 2, 200그루 | 0.4487 | 0.4594 | −0.0020 | 2/5 |
| depth 2, 300그루 | 0.4532 | 0.4595 | −0.0019 | 2/5 |
| depth 4, 100그루 | 0.4454 | 0.4562 | −0.0051 | 2/5 |
| depth 4, 150그루 | 0.4473 | 0.4582 | −0.0032 | 2/5 |
| depth 4, 200그루 | 0.4493 | 0.4560 | −0.0053 | 1/5 |
| **colsample 0.2, 200그루** | **0.4600** | **0.4641** | **+0.0028** | 3/5 |
| colsample 0.2, 300그루 | 0.4534 | 0.4580 | −0.0034 | 1/5 |
| colsample 0.5, 150그루 | 0.4488 | 0.4567 | −0.0046 | 1/5 |
| colsample 0.5, 200그루 | 0.4511 | 0.4577 | −0.0037 | 2/5 |

- 기준 200그루의 XGB 단독 CV 0.4489는 27 노트북과 같은 값 → 단계별 예측(`iteration_range`)이 올바르게 동작함을 확인
- 12개 설정 중 **채택 기준(4/5 fold + 상승폭 > std)을 넘는 것이 없음**
- 가장 좋은 colsample 0.2(200그루)는 XGB 단독이 +0.011 오르지만, LR과 섞으면 +0.0028(3/5)로 줄어듦 → fold 간 std(0.0094)보다 작음
- depth를 바꾸거나(2, 4) 나무 수를 늘리면(300) 섞기 점수가 모두 내려감
- 진단: colsample 0.2의 test 예측은 기준과 6.4% 다르고(OOF 10.7%), STES 21.4% → 20.6%, COAD 13.8% → 14.5%로 큰 변화 없음

## 결론
- **H28 기각**: 22b 피처에서도 10번에서 고른 설정(depth 3, colsample 0.3, 200그루)이 여전히 가장 좋은 쪽 → 설정을 바꿔서 얻을 것이 없음
- 지금 C의 XGB 설정을 그대로 유지. 제출 파일 없음
- 참고: colsample을 0.3보다 낮추는 방향(0.2)만 약한 상승 신호가 있었지만 채택 기준 미달. 12개를 비교해 가장 좋은 것을 고른 값이라 실제 효과는 더 작을 것
